In [45]:
import duckdb
from splink import Linker, DuckDBAPI, SettingsCreator, block_on
import splink.comparison_library as cl
import splink.comparison_level_library as cll

con = duckdb.connect()

df_l = con.read_parquet("./perturbed_data_l.parquet")
df_r = con.read_parquet("./perturbed_data_r.parquet")

db_api = DuckDBAPI(con)

df_l = db_api.register(df_l, dataset_display_name="df_l")
df_r = db_api.register(df_r, dataset_display_name="df_r")

In [50]:
linker = Linker(
    splink_dataframe_or_dataframes=[df_l, df_r],
    settings="./model.json",
    log_level="INFO",
)

df_predict = linker.inference.predict()

df_clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict=df_predict,
    threshold_match_probability=0.8,
)

# df_clusters = linker.clustering.cluster_using_single_best_links(
#     df_predict=df_predict,
#     threshold_match_probability=0.8,
#     duplicate_free_datasets=['df_l', 'df_r'],
# )

Blocking time: 3.07 seconds
Predict time (post-blocking): 9.03 seconds
Completed iteration 1, num edges remaining to process: 9488
Completed iteration 2, num edges remaining to process: 478
Completed iteration 3, num edges remaining to process: 24
Completed iteration 4, num edges remaining to process: 0


In [51]:
# investigate cluster sizes
cluster_sizes = con.sql(
    f"""
    select
        cluster_id,
        sum(case when source_dataset = 'df_l' then 1 else 0 end) as size_l,
        sum(case when source_dataset = 'df_r' then 1 else 0 end) as size_r
    from {df_clusters.physical_name}
    group by all
    """
)

con.sql(
    f"""
    select
        size_l,
        size_r,
        count(*) as count
    from cluster_sizes
    group by all
    order by count desc
    """
)

size_lHUGEINT,size_rHUGEINT,countBIGINT
1,1,64690
1,0,10918
0,1,10897
2,2,2780
1,2,2499
2,1,2496
0,2,289
3,3,283
3,2,274
2,0,267


In [52]:
# investigate person_ids which are split across two or more clusters (missed links)
splits = con.sql(
    f"""
    select
        person_id,
        count(distinct cluster_id) > 1 as split
    from {df_clusters.physical_name}
    group by person_id
    """
)

con.sql(
    f"""
    select 
        split,
        count(*)
    from splits
    group by split
    """
)

splitBOOLEAN,count_star()BIGINT
True,3077
False,95987


In [53]:
# investigate clusters with more than one person id (false links)
merges = con.sql(
    f"""
    select
        cluster_id,
        count(distinct person_id) > 1 as merged
    from {df_clusters.physical_name}
    group by cluster_id
    """
)

con.sql(
    f"""
    select 
        merged,
        count(*)
    from merges
    group by merged
    """
)

mergedBOOLEAN,count_star()BIGINT
False,90906
True,5117
